In [0]:
spark.sql("DROP TABLE IF EXISTS demo.UPI_raw_transactions")

In [0]:
# create a Catalog
spark.sql ( """
           CREATE TABLE IF NOT EXISTS demo.UPI_raw_transactions (
               Transaction_Id STRING,
               UPI_ID STRING,
               Merchant_ID STRING,
               Transaction_Amount DECIMAL(10,2),
               Transaction_Date TIMESTAMP,
               Transaction_Status STRING)
            USING DELTA
            TBLPROPERTIES("delta.enableChangeDataFeed" = "true")
           """)
print("Table demo.UPI_raw_transactions created")


In [0]:
import time
from delta.tables import DeltaTable


mock_data = [
    spark.createDataFrame([
        ("TRXN1", "UPI1", "M001", 1500.00, "2025-09-24 06:30:14", "initiated"),
        ("TRXN2", "UPI2", "M002", 900.90, "2025-09-24 06:30:19", "initiated"),
        ("TRXN3", "UPI3", "M002", 1200.00, "2025-09-24 06:30:34", "initiated"),
        ("TRXN4", "UPI4", "M003", 7000.00, "2025-09-24 06:30:44", "initiated")
    ], ["Transaction_Id", "UPI_ID", "Merchant_ID","Transaction_Amount", "Transaction_Date", "Transaction_Status"]),
    spark.createDataFrame([
        ("TRXN3", "UPI3", "M002", 1200.00, "2025-09-24 06:31:19", "Success"),
        ("TRXN4", "UPI4", "M003",7000.00, "2025-09-24 06:31:19", "Failed"),
        ("TRXN5", "UPI45", "M004",1200.00, "2025-09-24 06:31:34", "initiated"),
        ("TRXN6", "UPI94", "M004",7000.00, "2025-09-24 06:32:44", "initiated")
    ], ["Transaction_Id", "UPI_ID", "Merchant_ID","Transaction_Amount", "Transaction_Date", "Transaction_Status"]),
    spark.createDataFrame([
        ("TRXN1", "UPI1", "M001", 1500.00, "2025-09-24 06:33:14", "Success"),
        ("TRXN2", "UPI2", "M002", 900.90, "2025-09-24 06:33:19", "Success"),
        ("TRXN8", "UPI4", "M004", 7000.00, "2025-09-24 06:33:39", "initiated"),
        ("TRXN9", "UPI95", "M003",1200.00, "2025-09-24 06:34:34", "initiated"),
        ("TRXN10", "UPI1", "M003",10000.00, "2025-09-24 06:35:44", "initiated")
    ], ["Transaction_Id", "UPI_ID", "Merchant_ID", "Transaction_Amount", "Transaction_Date", "Transaction_Status"]),
    spark.createDataFrame([
        ("TRXN8", "UPI4", "M004",7000.00, "2025-09-24 06:39:39", "Success"),
        ("TRXN9", "UPI95", "M003",1200.00, "2025-09-24 06:40:34", "Refunded"),
        ("TRXN10", "UPI1", "M003",10000.00, "2025-09-24 06:40:44", "Failed")
    ], ["Transaction_Id", "UPI_ID", "Merchant_ID", "Transaction_Amount", "Transaction_Date", "Transaction_Status"])
]

def merge_data(delta_target_table, df):
    delta_table = DeltaTable.forName(spark, delta_target_table)
    delta_table.alias("target") \
        .merge(
            df.alias("source"),
            "target.Transaction_Id = source.Transaction_Id"
        ). whenMatchedUpdate(
            set = {
                "UPI_ID" : "source.UPI_ID",
                "Transaction_Amount": "source.Transaction_Amount",
                "Transaction_Date": "source.Transaction_Date",
                "Transaction_Status": "source.Transaction_Status"
            }
        ). whenNotMatchedInsertAll() \
        .execute()
    print('Merge completed')

for i in range(len(mock_data)):
     merge_data("demo.UPI_raw_transactions", mock_data[i])
     time.sleep(60)

#merge_data("demo.UPI_raw_transactions", mock_data[3])
